# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates loading and exploring the [FAIR² dataset](https://doi.org/10.71728/senscience.qs2f-h81p) using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library. The dataset's structure and records are defined using a [Croissant schema](https://mlcommons.org/croissant/).

### Dataset Source
The dataset source is defined by the following Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant (uncomment if not already installed)
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an object: access attributes directly

print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n\nLicense: {metadata.license}\nVersion: {metadata.version}\n")

## 2. Data Overview
Review available record sets and fields defined by their `@id`. All entities are referenced using their `@id` field per Croissant best practices.

In [ ]:
# List available record sets and their fields by @id
record_sets = []
print("Record sets present in this dataset:")
for record_set in dataset.record_sets:
    print(f"- RecordSet @id: {record_set['@id']}, name: {record_set.get('name', '<no name>')}")
    # List fields for each record set
    if 'field' in record_set:
        fields = record_set['field']
        if isinstance(fields, dict):
            fields = [fields]
        print("  Fields:")
        for field in fields:
            if isinstance(field, dict):
                print(f"    - Field @id: {field['@id']}, name: {field.get('name','<no name>')}, dataType: {field.get('dataType','')} ")
            else:
                print(f"    - Field @id: {field}")
    record_sets.append(record_set['@id'])

if not record_sets:
    print("No explicit record sets defined; attempting to discover record sets defined by the mlcroissant dataset object.")
    # Try to get from dataset API
    discovered_record_sets = dataset.record_set_ids
    for rsid in discovered_record_sets:
        print(f"- RecordSet @id: {rsid}")
    record_sets = discovered_record_sets

print("\nAll discovered record sets' @id: ", record_sets)

## 3. Data Extraction
Load the records from each record set (table) into a pandas DataFrame for exploration. Refer to record set and field `@id`s only.

In [ ]:
# Extract all records from each record set using mlcroissant, keyed by their @id
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
    except Exception as e:
        print(f"Could not load RecordSet @id {record_set_id}: {e}")

# Show columns and first rows from the primary record set (assume single table if only one)
if record_sets:
    primary_record_set = record_sets[0]
    df = dataframes[primary_record_set]
    print(f"\nColumns for RecordSet @id: {primary_record_set}")
    print(list(df.columns))
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering, normalizing, and grouping records. All field and record set references are by their `@id`.

In [ ]:
# Select a numeric field by @id. (Adjust the @id as needed per dataset structure)
# For illustration, let's suppose 'age' is a numeric field with @id 'https://api.app.sen.science/frontiers/7862866/age'.
numeric_field_id = None
# Find a likely numeric field from the dataframe columns
primary_df = dataframes[primary_record_set]
for col in primary_df.columns:
    if 'age' in col.lower() or 'interval' in col.lower() or primary_df[col].dtype.kind in {'i','u','f'}:
        numeric_field_id = col
        break

if numeric_field_id is None:
    raise ValueError("No numeric field detected to demonstrate EDA. Please check dataset schema.")

print(f"Using numeric field @id: {numeric_field_id}")

# Set a threshold (example: median)
threshold = primary_df[numeric_field_id].median() if pd.api.types.is_numeric_dtype(primary_df[numeric_field_id]) else 10

filtered_df = primary_df[primary_df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the selected field (z-score)
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Group by another available field (example: 'Sex', using its @id if available)
group_field_id = None
for col in primary_df.columns:
    if 'sex' in col.lower() or 'gender' in col.lower():
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
    display(grouped_df)
else:
    print("\nNo group field (e.g., 'Sex' or 'Gender') found for grouping.")

## 5. Visualization
Visualize data distributions or relationships using `matplotlib` or `seaborn`. All visualizations are based on data referenced by field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Visualize numeric_field distribution (histogram)
plt.figure(figsize=(8,5))
sns.histplot(primary_df[numeric_field_id], kde=True, bins=15)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# If group field detected, show boxplot
if group_field_id:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=primary_df[group_field_id], y=primary_df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, preprocess, and visualize a Croissant-structured clinical oncology dataset using only `@id` references for all data entities. Key steps included reviewing available record sets, normalizing numeric data, and exploring groupwise differences. For more advanced analytics or integration of additional record sets, repeat the procedures above by referencing their `@id`s.